In [1]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"

# datasets
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# policy
from lerobot.policies.act.modeling_act import ACTPolicy

# robots
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.cameras.configs import Cv2Rotation
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# record utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.record import record_loop

# torch
from torch import cuda

# utils
from dotenv import load_dotenv

# my code
from scripts.utils.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Set Params:

In [ ]:
PUSH_TO_HUB     = False
SAVE_TO_DATASET = False
REPO_NAME         = 'so101_table_leg_move' # must only if saving to dataset
EXPERIMENT_NAME   = '50_episodes_no_augmentations'
POLICY_CHECKPOINT = '060000'
POLICY_TYPE = 'act'
NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 60
EVAL_TYPE = 'real'

Log to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

Initialize Policy:

In [4]:
# Initialize the policy
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / POLICY_CHECKPOINT / 'pretrained_model'
policy = ACTPolicy.from_pretrained(pretrained_name_or_path = policy_path, local_files_only=True)

Loading weights from local directory


Robots Config

In [5]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30, rotation=Cv2Rotation.NO_ROTATION),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Build Dataset:

In [10]:
if SAVE_TO_DATASET:
    # configure the dataset features
    action_features = hw_to_dataset_features(robot.action_features, "action")
    obs_features = hw_to_dataset_features(robot.observation_features, "observation")
    dataset_features = {**action_features, **obs_features}
    
    # create dataset
    policy_eval_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / EVAL_TYPE
    
    kwargs = {
    "repo_id": f"{HF_NAME}/{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    "root"   : str(policy_eval_path)
    }
    
    if policy_eval_path.exists():
        dataset=LeRobotDataset(**kwargs)
    else:
        dataset = LeRobotDataset.create(
            **kwargs,
            fps=FPS,
            features=dataset_features,
            robot_type=robot.name,
            use_videos=True,
            image_writer_threads=4,
        )
        

In [ ]:
# Initialize the keyboard listener and rerun visualization
_, events = init_keyboard_listener()
_init_rerun(session_name="recording")

# Connect the robot
robot.connect()

Initialize loop kwargs:

In [ ]:
kwargs = {
    "robot": robot,
    "events": events,
    "fps": FPS,
    "policy": policy,
    "control_time_s": EPISODE_TIME_SEC,
    "display_data": True,
}

if SAVE_TO_DATASET:
    kwargs["dataset"] = dataset
    kwargs["single_task"] = dataset.meta.tasks

In [ ]:
for episode_idx in range(NUM_EPISODES):
    log_say(f"Running inference, recording eval episode {episode_idx + 1} of {NUM_EPISODES}")
    record_loop(**kwargs)
    
    if SAVE_TO_DATASET:
        dataset.save_episode()

In [ ]:
# Clean up
robot.disconnect()
dataset.push_to_hub()